运行约定：以本 Notebook 所在目录作为工作目录，从第一单元顺序运行。首个代码单元仅建立数据与输出目录。历史输出保留，不代表当前环境已验证。


In [ ]:
from pathlib import Path
Path('../../data').mkdir(parents=True, exist_ok=True)
Path('../../outputs/paddle').mkdir(parents=True, exist_ok=True)


In [5]:
# encoding=utf8
import io
import os
import sys
import requests
from collections import OrderedDict 
import math
import random
import numpy as np
import paddle
from paddle.nn import Embedding
import paddle.nn.functional as F
import paddle.nn as nn

import warnings
warnings.filterwarnings("ignore")

# 词向量：迈向NLP领域的第1步台阶


## 1. 词向量概述

在自然语言处理任务中，词向量（Word Embedding）是表示自然语言里单词的一种方法，即把每个词都表示为一个N维空间内的点，即一个高维空间内的向量。通过这种方法，实现把自然语言计算转换为向量计算。

如 **图1** 所示的词向量计算任务中，先把每个词（如queen，king等）转换成一个高维空间的向量，这些向量在一定意义上可以代表这个词的语义信息。再通过计算这些向量之间的距离，就可以计算出词语之间的关联关系，从而达到让计算机像计算数值一样去计算自然语言的目的。

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/00ba55f7304e4f97942165cf1deb946ced404a19325d40969b7c220e30cf527e" width="800" ></center>
<center>图1：词向量计算示意图</center>
<br></br>

因此，大部分词向量模型都需要回答两个问题：

1. **如何把词转换为向量?**

自然语言单词是离散信号，比如“香蕉”，“橘子”，“水果”在我们看来就是3个离散的词。

如何把每个离散的单词转换为一个向量？

2. **如何让向量具有语义信息?**

比如，我们知道在很多情况下，“香蕉”和“橘子”更加相似，而“香蕉”和“句子”就没有那么相似，同时“香蕉”和“食物”、“水果”的相似程度可能介于“橘子”和“句子”之间。

那么，我们该如何让词向量具备这样的语义信息？


### 1.1 如何把词转换为向量

自然语言单词是离散信号，比如“我”、“ 爱”、“人工智能”。如何把每个离散的单词转换为一个向量？通常情况下，我们可以维护一个如 **图2** 所示的查询表。表中每一行都存储了一个特定词语的向量值，每一列的第一个元素都代表着这个词本身，以便于我们进行词和向量的映射（如“我”对应的向量值为 [0.3，0.5，0.7，0.9，-0.2，0.03] ）。给定任何一个或者一组单词，我们都可以通过查询这个excel，实现把单词转换为向量的目的，这个查询和替换过程称之为Embedding Lookup。
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/2ccf57a0c4584ca5b5000da85bc48c4cbe1588f0e3f548c6b2271c26e2e7df14" width="800" ></center>
<center><br>图2：词向量查询表</br></center>
<br></br>

上述过程也可以使用一个字典数据结构实现。事实上如果不考虑计算效率，使用字典实现上述功能是个不错的选择。然而在进行神经网络计算的过程中，需要大量的算力，常常要借助特定硬件（如GPU）满足训练速度的需求。GPU上所支持的计算都是以张量（Tensor）为单位展开的，因此在实际场景中，我们需要把Embedding Lookup的过程转换为张量计算，如 **图3** 所示。
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/2145ed77e5b24e0b9b33a7c8451b52cdc68f38b651494b51afdd264724392fe3" width="800" ></center>
<center><br>图3：张量计算示意图</br></center>
<br></br>

假设对于句子"我，爱，人工，智能"，把Embedding Lookup的过程转换为张量计算的流程如下：

1. 通过查询字典，先把句子中的单词转换成一个ID（通常是一个大于等于0的整数），这个单词到ID的映射关系可以根据需求自定义（如**图3**中，我=>1, 人工=>2，爱=>3，...）。

2. 得到ID后，再把每个ID转换成一个固定长度的向量。假设字典的词表中有5000个词，那么，对于单词“我”，就可以用一个5000维的向量来表示。由于“我”的ID是1，因此这个向量的第一个元素是1，其他元素都是0（[1，0，0，…，0]）；同样对于单词“人工”，第二个元素是1，其他元素都是0。用这种方式就实现了用一个向量表示一个单词。由于每个单词的向量表示都只有一个元素为1，而其他元素为0，因此我们称上述过程为One-Hot Encoding。

3. 经过One-Hot Encoding后，句子“我，爱，人工，智能”就被转换成为了一个形状为 4×5000的张量，记为$V$。在这个张量里共有4行、5000列，从上到下，每一行分别代表了“我”、“爱”、“人工”、“智能”四个单词的One-Hot Encoding。最后，我们把这个张量$V$和另外一个稠密张量$W$相乘，其中$W$张量的形状为5000 × 128（5000表示词表大小，128表示每个词的向量大小）。经过张量乘法，我们就得到了一个4×128的张量，从而完成了把单词表示成向量的目的。

### 1.2 如何让向量具有语义信息

得到每个单词的向量表示后，我们需要思考下一个问题：比如在多数情况下，“香蕉”和“橘子”更加相似，而“香蕉”和“句子”就没有那么相似；同时，“香蕉”和“食物”、“水果”的相似程度可能介于“橘子”和“句子”之间。那么如何让存储的词向量具备这样的语义信息呢？

我们先学习自然语言处理领域的一个小技巧。在自然语言处理研究中，科研人员通常有一个共识：使用一个单词的上下文来了解这个单词的语义，比如：

 >“苹果手机质量不错，就是价格有点贵。”
 >
 >“这个苹果很好吃，非常脆。”
 >
 >“菠萝质量也还行，但是不如苹果支持的APP多。”
>
在上面的句子中，我们通过上下文可以推断出第一个“苹果”指的是苹果手机，第二个“苹果”指的是水果苹果，而第三个“菠萝”指的应该也是一个手机。事实上，在自然语言处理领域，使用上下文描述一个词语或者元素的语义是一个常见且有效的做法。我们可以使用同样的方式训练词向量，让这些词向量具备表示语义信息的能力。

## 2. word2vec 词向量训练算法

2013年，Mikolov提出的经典word2vec算法就是通过上下文来学习语义信息。word2vec包含两个经典模型：CBOW（Continuous Bag-of-Words）和Skip-gram，如 **图4** 所示。

- **CBOW**：通过上下文的词向量推理中心词。
- **Skip-gram**：根据中心词推理上下文。

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/87b90136eef04d7285803e567a5f7f6d0e40bb552deb40a19a7540c4e6aa20a3" width="700" ></center>
<center><br>图4：CBOW和Skip-gram语义学习示意图</br></center>
<br></br>

假设有一个句子“Pineapples are spiked and yellow”，两个模型的推理方式如下：

- 在**CBOW**中，先在句子中选定一个中心词，并把其它词作为这个中心词的上下文。如 **图4** CBOW所示，把“Spiked”作为中心词，把“Pineapples、are、and、yellow”作为中心词的上下文。在学习过程中，使用上下文的词向量推理中心词，这样中心词的语义就被传递到上下文的词向量中，如“Spiked → pineapple”，从而达到学习语义信息的目的。

- 在**Skip-gram**中，同样先选定一个中心词，并把其他词作为这个中心词的上下文。如 **图4** Skip-gram所示，把“Spiked”作为中心词，把“Pineapples、are、and、yellow”作为中心词的上下文。不同的是，在学习过程中，使用中心词的词向量去推理上下文，这样上下文定义的语义被传入中心词的表示中，如“pineapple → Spiked”，
从而达到学习语义信息的目的。

------
**说明：**

一般来说，CBOW比Skip-gram训练速度快，训练过程更加稳定，原因是CBOW使用上下文average的方式进行训练，每个训练step会见到更多样本。而在生僻字（出现频率低的字）处理上，skip-gram比CBOW效果更好，原因是skip-gram不会刻意回避生僻字(CBOW结构中输入中存在生僻字时，生僻字会被其它非生僻字的权重冲淡)。

------

## 4. 使用PaddleNLP加载词向量

PaddleNLP已经内置了多个开源的预训练词向量模型，用户仅需在使用paddlenlp.embeddings.TokenEmbedding时，指定预训练模型的名称，即可加载相对应的预训练模型。以下将介绍TokenEmbeddign详细用法，并列出PaddleNLP所支持的预训练Embedding模型。

| 参数 | 类型 | 属性 |
| -------- | -------- | -------- |
| embedding_name     | String     | 预训练embedding名称，可通过paddlenlp.embeddings.list_embedding_name()查询。     |
|unknown_token|string|unknown token。
|unknown_token_vector|list 或者 np.array|用来初始化unknown token对应的vector。默认为None（以正态分布方式初始化vector）
|extended_vocab_path|string|扩展词表的文件名路径。词表格式为一行一个词。
|trainable|bool|是否可训练。True表示Embedding可以更新参数，False为不可更新。

### 4.1 加载word2vec词向量

下面我们尝试列出PaddleNLP内置的Embedding，并基于TokenEmbedding API 加载词向量`w2v.baidu_encyclopedia.target.word-word.dim300`，该词向量中每个单词的词向量维度均为300。相关代码如下：



In [6]:
import paddle
from paddlenlp.embeddings import TokenEmbedding, list_embedding_name
paddle.set_device("gpu:0")

# 查看预训练embedding模型名称：
print(list_embedding_name()) 

# 初始化TokenEmbedding， 预训练embedding没下载时会自动下载并加载数据
# 默认缓存路径： 
token_embedding = TokenEmbedding(embedding_name="w2v.baidu_encyclopedia.target.word-word.dim300")


# 查看token_embedding详情
print(token_embedding)

[2026-01-07 16:54:51,059] [    INFO] - Loading token embedding...


['w2v.baidu_encyclopedia.target.word-word.dim300', 'w2v.baidu_encyclopedia.target.word-character.char1-1.dim300', 'w2v.baidu_encyclopedia.target.word-character.char1-2.dim300', 'w2v.baidu_encyclopedia.target.word-character.char1-4.dim300', 'w2v.baidu_encyclopedia.target.word-ngram.1-2.dim300', 'w2v.baidu_encyclopedia.target.word-ngram.1-3.dim300', 'w2v.baidu_encyclopedia.target.word-ngram.2-2.dim300', 'w2v.baidu_encyclopedia.target.word-wordLR.dim300', 'w2v.baidu_encyclopedia.target.word-wordPosition.dim300', 'w2v.baidu_encyclopedia.target.bigram-char.dim300', 'w2v.baidu_encyclopedia.context.word-word.dim300', 'w2v.baidu_encyclopedia.context.word-character.char1-1.dim300', 'w2v.baidu_encyclopedia.context.word-character.char1-2.dim300', 'w2v.baidu_encyclopedia.context.word-character.char1-4.dim300', 'w2v.baidu_encyclopedia.context.word-ngram.1-2.dim300', 'w2v.baidu_encyclopedia.context.word-ngram.1-3.dim300', 'w2v.baidu_encyclopedia.context.word-ngram.2-2.dim300', 'w2v.baidu_encyclopedi

[2026-01-07 16:54:54,171] [    INFO] - Finish loading embedding vector.
[2026-01-07 16:54:54,172] [    INFO] - Token Embedding info:             
Unknown index: 635963             
Unknown token: [UNK]             
Padding index: 635964             
Padding token: [PAD]             
Shape :[635965, 300]


Object   type: TokenEmbedding(635965, 300, padding_idx=635964, sparse=False)             
Unknown index: 635963             
Unknown token: [UNK]             
Padding index: 635964             
Padding token: [PAD]             
Parameter containing:
Tensor(shape=[635965, 300], dtype=float32, place=Place(gpu:0), stop_gradient=False,
       [[-0.24200200,  0.13931701,  0.07378800, ...,  0.14103900,
          0.05592300, -0.08004800],
        [-0.08671700,  0.07770800,  0.09515300, ...,  0.11196400,
          0.03082200, -0.12893000],
        [-0.11436500,  0.12201900,  0.02833000, ...,  0.11068700,
          0.03607300, -0.13763499],
        ...,
        [ 0.02628800, -0.00008300, -0.00393500, ...,  0.00654000,
          0.00024600, -0.00662600],
        [-0.02603570,  0.02309024,  0.04830394, ...,  0.03472544,
         -0.00550382, -0.00777404],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ]])


可以看到词向量的大小是[635965, 300]，其中635965表示的是词汇表的大小，300表示的是每个词使用的是300维的向量表示。



### 4.2 基于词向量进行计算

#### 4.2.1 检索词向量

在加载好PaddleNLP内置的词向量后， 可以通过如下方法，检索具体单词的词向量。

In [7]:
test_token_embedding = token_embedding.search("中国")
print(test_token_embedding)

[[ 0.260801  0.1047    0.129453 -0.257317 -0.16152   0.19567  -0.074868
   0.361168  0.245882 -0.219141 -0.388083  0.235189  0.029316  0.154215
  -0.354343  0.017746  0.009028  0.01197  -0.121429  0.096542  0.009255
   0.039721  0.363704 -0.239497 -0.41168   0.16958   0.261758  0.022383
  -0.053248 -0.000994 -0.209913 -0.208296  0.197332 -0.3426   -0.162112
   0.134557 -0.250201  0.431298  0.303116  0.517221  0.243843  0.022219
  -0.136554 -0.189223  0.148563 -0.042963 -0.456198  0.14546  -0.041207
   0.049685  0.20294   0.147355 -0.206953 -0.302796 -0.111834  0.128183
   0.289539 -0.298934 -0.096412  0.063079  0.324821 -0.144471  0.052456
   0.088761 -0.040925 -0.103281 -0.216065 -0.200878 -0.100664  0.170614
  -0.355546 -0.062115 -0.52595  -0.235442  0.300866 -0.521523 -0.070713
  -0.331768  0.023021  0.309111 -0.125696  0.016723 -0.0321   -0.200611
   0.057294 -0.128891 -0.392886  0.423002  0.282569 -0.212836  0.450132
   0.067604 -0.124928 -0.294086  0.136479  0.091505 -0.061723 -0

也可以向search API中传入一批单词，同时获取这些单词的词向量：


In [12]:
words = ['飞桨', '是', '优秀', '的', '深度', '学习', '平台']
test_token_embedding = token_embedding.search(words)

# 输出词向量的维度
print(test_token_embedding.shape)

(7, 300)


#### 4.2.2 计算词向量cosine相似度

计算词向量内积,使用token_embedding.dot API 可以轻松地计算两个词汇向量之间的内积

In [13]:
score = token_embedding.dot("中国", "美国")
print(score) 

8.611071




使用token_embedding.cosine_sim API 可以轻松地计算两个词汇之间的cosine余弦相似度，  
专门衡量方向的相似性，它通过将内积结果除以两个向量的模长来消除向量长度的影响

In [14]:
score = token_embedding.cosine_sim("中国", "美国")
print(score) 


0.49586025


#### 4.2.4 词向量可视化

一般两个单词的语义越相似，其词向量之间的距离越近，否则就越远。 我们可以获取前1000个词向量，然后使用深度学习可视化工具[VisualDL](https://github.com/PaddlePaddle/VisualDL)的High Dimensional组件对embedding结果进行可视化展示。相关代码如下：



In [11]:
# 引入VisualDL的LogWriter记录日志
from visualdl import LogWriter

# 获取词表中前1000个单词
labels = token_embedding.vocab.to_tokens(list(range(0,1000)))
test_token_embedding = token_embedding.search(labels)

with LogWriter(logdir='../../outputs/paddle/visualize') as writer:
    writer.add_embeddings(tag='test', mat=test_token_embedding, metadata=labels)

执行完毕后会在当前路径下生成一个visualize目录，并将日志存放在其中，我们在命令行启动VisualDL即可进行查看，启动命令为：

> visualdl --logdir ../../outputs/paddle/visualize

启动后打开浏览器即可看到可视化结果


**备注**：词向量可视化的操作，请在本地开发环境进行尝试。


<center><img src="https://ai-studio-static-online.cdn.bcebos.com/541186381e5f4d75bd8da5d9de748748631e4bb7e8314050bc7ea58029e07383" /></center>
<center>图7 词向量可视化</center>

### 4.3 基于PaddleNLP内置词向量计算句子间的相似度

前边讲了如何使用PaddleNLP内置的词向量进行词向量级的计算，那么我们应该如何使用这些词向量计算两个句子之间的相似度呢。

计算句子间的相似度的一般做法是先获取句子的句向量表示，然后根据句向量进行相似度计算，如cosine相似度。

#### 4.3.1 句向量表示
对于给定的一句话，应该如何获取一个向量对这句话进行表示呢。一种简单的做法是首先对给定的语句进行分词，然后将每个词的词向量进行相加，然后将向量平均作为这句话的句向量。

例如， 给定语句："飞桨是优秀的深度学习平台"，可以通过执行如下操作获取句向量：
- 语句分词：['飞桨', '是', '优秀', '的', '深度', '学习', '平台']
- 获取每个单词的词向量$V = [v_1, v_2, v_3, v_4, v_5, v_6, v_7]$
- 将词向量进行相加 $v = \sum_1^7 v_i$
- 将获取的向量$v$进行平均得到句向量： $v = v/7$


那么如何对一句话进行分词呢，这里可以使用PaddleNLP内置的分词组件JiebaTokenizer，需要注意的是，这里需要将加载的词向量的词表传给JiebaTokenizer，以帮助JiebaTokenizer进行分词，相关代码如下。


In [15]:

from paddlenlp.data import JiebaTokenizer

tokenizer = JiebaTokenizer(vocab=token_embedding.vocab)
tokenizer.cut('飞桨是优秀的深度学习平台')



['飞桨', '是', '优秀', '的', '深度', '学习', '平台']

下面，我们实现函数get_sentence_embedding函数，用以计算给定的语句的句向量，相关代码如下：

In [16]:
import numpy as np

def get_sentence_embedding(text):
    # 分词
    words = tokenizer.cut(text)
    # 获取词向量
    word_embeddings = token_embedding.search(words)
    # 计算句向量
    sentence_embedding = np.sum(word_embeddings, axis=0) / len(words)
    
    return sentence_embedding


text = "飞桨是优秀的深度学习平台"
get_sentence_embedding(text)

array([ 0.18727861,  0.14638546,  0.05162099, -0.16003159, -0.10168965,
        0.07344915,  0.15221961, -0.04196823,  0.22641562,  0.05097622,
        0.12351739, -0.0158484 ,  0.21031806,  0.08221895, -0.1569034 ,
       -0.10162197,  0.20115264,  0.11382335, -0.12669078,  0.00796119,
        0.06720693, -0.03313528,  0.04532251, -0.05308977, -0.09080763,
        0.14316188,  0.12221444,  0.24970753,  0.03657195,  0.040875  ,
        0.11211652,  0.04111849,  0.0842277 ,  0.07654665,  0.10827108,
        0.21127388,  0.07312528,  0.16625953,  0.11903051,  0.05286885,
        0.10440012, -0.1710639 , -0.09652049, -0.15704307,  0.15791057,
       -0.04051816, -0.11804091,  0.12072999,  0.10002882, -0.11529361,
        0.21839754,  0.06980227, -0.16191995, -0.04186095,  0.06080762,
       -0.04774191,  0.12031809, -0.00172876, -0.07859348, -0.0565686 ,
        0.17847247, -0.01730455,  0.05790719,  0.16849597, -0.00051663,
       -0.0825384 , -0.12015433,  0.03010257,  0.13332961,  0.08

#### 4.3.2 计算语句相似度

下面给出两句话text1和text2，分别获取两句话的句向量，然后计算两句话的cosine相似度，相关代码如下。

In [28]:
import paddle.nn.functional as F

text1 = "飞桨是优秀的深度学习平台"
text2 = "飞桨是一个深度学习平台"

sen_emb1 = get_sentence_embedding(text1)
sen_emb2 = get_sentence_embedding(text2)

sim = F.cosine_similarity(paddle.to_tensor(sen_emb1).unsqueeze(0), paddle.to_tensor(sen_emb2).unsqueeze(0))
print("Similarity: {:.5f}".format(sim.item()))


Similarity: 0.92731


In [27]:
import paddle.nn.functional as F

text1 = "飞桨是优秀的深度学习平台"
text2 = "PyTorch一个深度学习框架"

sen_emb1 = get_sentence_embedding(text1)
sen_emb2 = get_sentence_embedding(text2)

sim = F.cosine_similarity(paddle.to_tensor(sen_emb1).unsqueeze(0), paddle.to_tensor(sen_emb2).unsqueeze(0))
print("Similarity: {:.5f}".format(sim.item()))


Similarity: 0.73698


## 5. 常见word2vec预训练模型

### 5.1 英文预训练模型

| 模型                       | 语料                                      | 维度                   | 词汇量        | 下载地址                                                                                 |
| ------------------------ | --------------------------------------- | -------------------- | ---------- | ------------------------------------------------------------------------------------ |
| **Google News Word2Vec** | Google News (约 100B 词)                  | 300                  | \~300万词/短语 | [GoogleNews-vectors-negative300.bin.gz](https://code.google.com/archive/p/word2vec/) |
| **GloVe**                | Wikipedia + Gigaword / Common Crawl     | 50 / 100 / 200 / 300 | 数十亿词       | [GloVe 官方](https://nlp.stanford.edu/projects/glove/)                                 |
| **FastText (English)**   | Wikipedia / Common Crawl / UMBC webbase | 300                  | 数百万词       | [FastText 官方](https://fasttext.cc/docs/en/english-vectors.html)                      |


### 5.2 中文预训练模型

| 模型                         | 语料                | 维度        | 词汇量   | 下载地址                                                                 |
| -------------------------- | ----------------- | --------- | ----- | -------------------------------------------------------------------- |
| **腾讯词向量 (Tencent AI Lab)** | 海量中文互联网语料         | 200       | 800万  | [Tencent Embedding](https://ai.tencent.com/ailab/nlp/embedding.html) |
| **北大中文 Word2Vec**          | 中文 Wikipedia + 新闻 | 300       | 数十万   | GitHub 第三方整理版                                                        |
| **中文 Wikipedia Word2Vec**  | 中文 Wikipedia      | 100 / 300 | \~几十万 | [部分 GitHub 资源](https://github.com/Embedding/Chinese-Word-Vectors)    |
| **FastText 中文**            | Wikipedia / 新闻语料  | 300       | 数百万   | [FastText 中文](https://fasttext.cc/docs/en/crawl-vectors.html)        |
